# Cross-Modal Representational Alignment: Embedding Extraction Pipeline

This notebook extracts layer-wise embeddings from vision, audio, and language foundation models for use in RSA and Linear Predictivity analyses.

**Models**
- Vision: DINOv2 Base (`facebook/dinov2-base`) and Large (`facebook/dinov2-large`)
- Audio: BEATs Iter3 and Iter3+ (Microsoft UniLM)
- Language: Qwen3-1.7B (`Qwen/Qwen3-1.7B`)

**Inputs required**
- `final_audiocaps.csv` — filtered dataset (~620 samples) with `audiocap_id` and `caption` columns
- `images/middle_frames/` — middle video frames as `{audiocap_id}.png`
- `images/generated_images/` — SDXL-generated images as `{audiocap_id}/0.png` (or upload `generated_images.zip`)
- `audio/` — audio clips as `{audiocap_id}.wav`
- `beats/BEATs_iter3.pt` and `beats/BEATs_iter3_plus_AS2M.pt` — BEATs checkpoints (download manually)

**Outputs**
Layer-wise `.npy` files for each model, organized under `dinov2/`, `beats/`, and `qwen3-1.7b/`.


## 1. Setup

In [ ]:
# Install / clone BEATs (Microsoft UniLM)
!git clone https://github.com/microsoft/unilm.git

import sys
sys.path.append("/content/unilm/beats")
from BEATs import BEATs, BEATsConfig


In [ ]:
import os
import gc
import shutil

import numpy as np
import pandas as pd
import torch
import torchaudio
from PIL import Image
from tqdm import tqdm
from transformers import (
    AutoImageProcessor,
    AutoModel,
    AutoTokenizer,
    AutoModelForCausalLM,
)

print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# Create directory structure
for path in [
    "images/middle_frames",
    "images/generated_images",
    "audio",
    "beats",
    "dinov2/large/frame_embeddings", "dinov2/large/frame_layers",
    "dinov2/large/gen_embeddings",   "dinov2/large/gen_layers",
    "dinov2/base/frame_embeddings",  "dinov2/base/frame_layers",
    "dinov2/base/gen_embeddings",    "dinov2/base/gen_layers",
    "beats/iter3/audio_embeddings",  "beats/iter3/audio_layers",
    "beats/iter3_plus/audio_embeddings", "beats/iter3_plus/audio_layers",
    "qwen3-1.7b/text_embeddings",    "qwen3-1.7b/text_layers",
    "results",
]:
    os.makedirs(path, exist_ok=True)


In [ ]:
# Upload files manually in Colab, or adjust paths for your environment:
#   - final_audiocaps.csv
#   - images/middle_frames/{audiocap_id}.png
#   - generated_images.zip  (will be unzipped below)
#   - audio/{audiocap_id}.wav
#   - beats/BEATs_iter3.pt
#   - beats/BEATs_iter3_plus_AS2M.pt

# Unzip generated images if uploaded as a zip
if os.path.exists("generated_images.zip"):
    shutil.unpack_archive("generated_images.zip", "unzipped_gen_images", "zip")
    src = "unzipped_gen_images/generated_images"
    if os.path.exists(src):
        shutil.move(src, "images/generated_images")
    print("Generated images unzipped.")


In [ ]:
df = pd.read_csv("final_audiocaps.csv")
print(f"Dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


## 2. Embedding Extraction Functions

Helper functions for DINOv2 (vision), BEATs (audio), and Qwen3 (language).
Each function extracts per-layer hidden states and saves one `.npy` file per sample,
then `get_layer_embeddings` reorganizes them into per-layer matrices across all samples.


In [ ]:
# ── DINOv2 ──────────────────────────────────────────────────────────────────

def load_dinov2(model_size="large", device=None):
    """Load a DINOv2 model and its processor."""
    model_map = {
        "small": "facebook/dinov2-small",
        "base":  "facebook/dinov2-base",
        "large": "facebook/dinov2-large",
        "giant": "facebook/dinov2-giant",
    }
    if model_size not in model_map:
        raise ValueError(f"model_size must be one of {list(model_map.keys())}")
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model_name = model_map[model_size]
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()
    return processor, model, device


def extract_dinov2_embeddings(
    df, image_root, save_root, processor, model, device,
    pooling="mean", gen_image=True, overwrite=False
):
    """
    Extract layer-wise DINOv2 embeddings and save one .npy per sample.

    Args:
        df:         DataFrame with an `audiocap_id` column.
        image_root: Root directory containing images.
        save_root:  Directory to save per-sample .npy files.
        processor:  HuggingFace image processor.
        model:      DINOv2 model.
        device:     Torch device.
        pooling:    'mean' (average patch tokens) or 'cls' (CLS token).
        gen_image:  True if images are stored as {audiocap_id}/0.png,
                    False if stored as {audiocap_id}.png.
        overwrite:  Re-extract even if the file already exists.
    """
    os.makedirs(save_root, exist_ok=True)

    for _, row in tqdm(df.iterrows(), total=len(df)):
        image_id = str(row["audiocap_id"])
        img_path = (
            os.path.join(image_root, image_id, "0.png") if gen_image
            else os.path.join(image_root, f"{image_id}.png")
        )
        save_path = os.path.join(save_root, f"{image_id}.npy")

        if not os.path.exists(img_path):
            print(f"Missing image: {img_path}")
            continue
        if os.path.exists(save_path) and not overwrite:
            continue

        image = Image.open(img_path).convert("RGB")
        inputs = {k: v.to(device) for k, v in processor(images=image, return_tensors="pt").items()}

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            if pooling == "cls":
                layer_embeddings = torch.stack([h[:, 0, :] for h in outputs.hidden_states])
            elif pooling == "mean":
                layer_embeddings = torch.stack([h.mean(dim=1) for h in outputs.hidden_states])
            else:
                raise ValueError("pooling must be 'cls' or 'mean'")
            embedding = layer_embeddings.squeeze().cpu().numpy()

        np.save(save_path, embedding)

    print("DINOv2 embedding extraction complete.")


In [ ]:
# ── BEATs ───────────────────────────────────────────────────────────────────

def load_beats_model(checkpoint_path, device, weights_only=True):
    """Load a BEATs model from a checkpoint file."""
    try:
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=weights_only)
    except TypeError:
        # older PyTorch versions don't support weights_only
        checkpoint = torch.load(checkpoint_path, map_location=device)

    cfg = BEATsConfig(checkpoint["cfg"])
    model = BEATs(cfg)
    model.load_state_dict(checkpoint["model"])
    model = model.to(device)
    model.eval()
    return model


def extract_beats_embeddings(
    model, df, audio_root, save_root, device,
    target_sr=16000, segment_sec=10
):
    """
    Extract layer-wise BEATs embeddings using forward hooks.

    Captures the output of each transformer encoder layer, mean-pools over
    time, and saves one .npy per sample with shape (num_layers, hidden_dim).

    Args:
        model:       BEATs model.
        df:          DataFrame with an `audiocap_id` column.
        audio_root:  Directory containing {audiocap_id}.wav files.
        save_root:   Directory to save per-sample .npy files.
        device:      Torch device.
        target_sr:   Resample audio to this sample rate.
        segment_sec: Seconds of audio centered on the midpoint to use.
    """
    os.makedirs(save_root, exist_ok=True)

    hidden_states = []

    def hook_fn(module, input, output):
        out = output[0] if isinstance(output, tuple) else output
        hidden_states.append(out.detach().cpu())

    handles = [layer.register_forward_hook(hook_fn) for layer in model.encoder.layers]
    segment_len = segment_sec * target_sr

    try:
        for _, row in tqdm(df.iterrows(), total=len(df)):
            audio_id = str(row["audiocap_id"])
            audio_path = os.path.join(audio_root, f"{audio_id}.wav")
            save_path  = os.path.join(save_root,  f"{audio_id}.npy")

            if not os.path.exists(audio_path):
                print(f"Missing audio: {audio_path}")
                continue
            if os.path.exists(save_path):
                continue

            waveform, sr = torchaudio.load(audio_path)
            waveform = waveform.mean(dim=0)  # stereo → mono
            if sr != target_sr:
                waveform = torchaudio.transforms.Resample(sr, target_sr)(waveform)

            # crop a segment centered on the midpoint
            mid = waveform.shape[0] // 2
            waveform = waveform[max(0, mid - segment_len // 2): mid + segment_len // 2]
            waveform = waveform.to(device)

            hidden_states.clear()
            with torch.no_grad():
                model.extract_features(waveform.unsqueeze(0))

            if not hidden_states:
                print(f"No hidden states captured for {audio_id}")
                continue

            # mean-pool over time for each layer → shape (num_layers, hidden_dim)
            layer_embeddings = torch.stack([h.squeeze(0).mean(dim=0) for h in hidden_states])
            np.save(save_path, layer_embeddings.numpy().astype(np.float32))
    finally:
        for h in handles:
            h.remove()

    print("BEATs embedding extraction complete.")


In [ ]:
# ── Qwen3 ───────────────────────────────────────────────────────────────────

def load_qwen3_model(model_name="Qwen/Qwen3-1.7B"):
    """Load Qwen3 model and tokenizer."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        device_map="auto",
    )
    model.eval()
    return tokenizer, model


def extract_qwen3_embeddings(
    df, text_col, save_root, tokenizer, model, device,
    pooling="mean", overwrite=False
):
    """
    Extract layer-wise Qwen3 embeddings and save one .npy per sample.

    Args:
        df:        DataFrame with `audiocap_id` and a text column.
        text_col:  Name of the column containing captions.
        save_root: Directory to save per-sample .npy files.
        tokenizer: HuggingFace tokenizer.
        model:     Qwen3 model.
        device:    Torch device.
        pooling:   'mean' (attention-masked mean), 'cls', or 'last' token.
        overwrite: Re-extract even if the file already exists.
    """
    os.makedirs(save_root, exist_ok=True)

    for _, row in tqdm(df.iterrows(), total=len(df)):
        text_id   = str(row["audiocap_id"])
        text      = row[text_col]
        save_path = os.path.join(save_root, f"{text_id}.npy")

        if os.path.exists(save_path) and not overwrite:
            continue

        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True, return_dict=True)
            hidden_states = outputs.hidden_states
            mask = inputs["attention_mask"]

            if pooling == "mean":
                layer_embeddings = torch.stack([
                    (h * mask.unsqueeze(-1)).sum(dim=1) / mask.sum(dim=1, keepdim=True)
                    for h in hidden_states
                ])
            elif pooling == "cls":
                layer_embeddings = torch.stack([h[:, 0, :] for h in hidden_states])
            elif pooling == "last":
                layer_embeddings = torch.stack([h[:, -1, :] for h in hidden_states])
            else:
                raise ValueError("pooling must be 'mean', 'cls', or 'last'")

            embedding = layer_embeddings.squeeze(1).cpu().float().numpy()

        np.save(save_path, embedding)

    print("Qwen3 embedding extraction complete.")


In [ ]:
# ── Layer matrix builder ─────────────────────────────────────────────────────

def get_layer_embeddings(input_folder, save_root, num_layers, emb_shape):
    """
    Reshape per-sample embeddings into per-layer matrices and save as .npy files.

    Reads all {audiocap_id}.npy files from `input_folder`, validates their shape,
    stacks them by layer, and saves layer_{i}.npy to `save_root`.
    Also saves ids.npy to record the sample order.

    Args:
        input_folder: Folder containing per-sample .npy files.
        save_root:    Folder to write per-layer .npy files.
        num_layers:   Expected number of layers.
        emb_shape:    Expected shape of each per-sample embedding (num_layers, hidden_dim).
    """
    os.makedirs(save_root, exist_ok=True)

    files = sorted([f for f in os.listdir(input_folder) if f.endswith(".npy") and f != "ids.npy"])
    ids   = [f.replace(".npy", "") for f in files]
    np.save(os.path.join(save_root, "ids.npy"), np.array(ids))

    for l in range(num_layers):
        print(f"Processing layer {l}", end="\r")
        layer_vectors = []
        for f in files:
            emb = np.load(os.path.join(input_folder, f))
            if emb.shape != emb_shape:
                raise ValueError(f"{f}: expected shape {emb_shape}, got {emb.shape}")
            layer_vectors.append(emb[l])
        np.save(os.path.join(save_root, f"layer_{l}.npy"), np.stack(layer_vectors).astype(np.float32))

    print(f"\nLayer embeddings saved to {save_root}  ({num_layers} layers, {len(files)} samples)")


## 3. Vision Embeddings (DINOv2)

Extract embeddings for both **middle frames** (real video frames) and **generated images** (SDXL output),
using DINOv2 Large (25 layers, 1024-dim) and DINOv2 Base (13 layers, 768-dim).


In [ ]:
# ── DINOv2 Large ─────────────────────────────────────────────────────────────
processor_large, model_large, device = load_dinov2(model_size="large", device=device)

# Middle frames
extract_dinov2_embeddings(
    df, image_root="images/middle_frames",
    save_root="dinov2/large/frame_embeddings",
    processor=processor_large, model=model_large, device=device,
    pooling="mean", gen_image=False,
)
get_layer_embeddings(
    input_folder="dinov2/large/frame_embeddings",
    save_root="dinov2/large/frame_layers",
    num_layers=25, emb_shape=(25, 1024),
)

# Generated images
extract_dinov2_embeddings(
    df, image_root="images/generated_images",
    save_root="dinov2/large/gen_embeddings",
    processor=processor_large, model=model_large, device=device,
    pooling="mean", gen_image=True,
)
get_layer_embeddings(
    input_folder="dinov2/large/gen_embeddings",
    save_root="dinov2/large/gen_layers",
    num_layers=25, emb_shape=(25, 1024),
)

del model_large; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# ── DINOv2 Base ──────────────────────────────────────────────────────────────
processor_base, model_base, device = load_dinov2(model_size="base", device=device)

# Middle frames
extract_dinov2_embeddings(
    df, image_root="images/middle_frames",
    save_root="dinov2/base/frame_embeddings",
    processor=processor_base, model=model_base, device=device,
    pooling="mean", gen_image=False,
)
get_layer_embeddings(
    input_folder="dinov2/base/frame_embeddings",
    save_root="dinov2/base/frame_layers",
    num_layers=13, emb_shape=(13, 768),
)

# Generated images
extract_dinov2_embeddings(
    df, image_root="images/generated_images",
    save_root="dinov2/base/gen_embeddings",
    processor=processor_base, model=model_base, device=device,
    pooling="mean", gen_image=True,
)
get_layer_embeddings(
    input_folder="dinov2/base/gen_embeddings",
    save_root="dinov2/base/gen_layers",
    num_layers=13, emb_shape=(13, 768),
)

del model_base; gc.collect(); torch.cuda.empty_cache()


## 4. Audio Embeddings (BEATs)

Extract embeddings using BEATs Iter3 and Iter3+ (12 transformer layers, 768-dim each).
Forward hooks capture the output of each encoder layer; embeddings are mean-pooled over time.

> **Note:** BEATs checkpoints must be downloaded manually and placed in `beats/`:
> - [`BEATs_iter3.pt`](https://valle.blob.core.windows.net/share/BEATs/BEATs_iter3.pt)
> - [`BEATs_iter3_plus_AS2M.pt`](https://valle.blob.core.windows.net/share/BEATs/BEATs_iter3_plus_AS2M.pt)


In [ ]:
# ── BEATs Iter3+ ─────────────────────────────────────────────────────────────
beats3_plus = load_beats_model("beats/BEATs_iter3_plus_AS2M.pt", device=device)
extract_beats_embeddings(
    model=beats3_plus, df=df,
    audio_root="audio",
    save_root="beats/iter3_plus/audio_embeddings",
    device=device,
)
get_layer_embeddings(
    input_folder="beats/iter3_plus/audio_embeddings",
    save_root="beats/iter3_plus/audio_layers",
    num_layers=12, emb_shape=(12, 768),
)
del beats3_plus; gc.collect(); torch.cuda.empty_cache()

# ── BEATs Iter3 ──────────────────────────────────────────────────────────────
beats3 = load_beats_model("beats/BEATs_iter3.pt", device=device, weights_only=False)
extract_beats_embeddings(
    model=beats3, df=df,
    audio_root="audio",
    save_root="beats/iter3/audio_embeddings",
    device=device,
)
get_layer_embeddings(
    input_folder="beats/iter3/audio_embeddings",
    save_root="beats/iter3/audio_layers",
    num_layers=12, emb_shape=(12, 768),
)
del beats3; gc.collect(); torch.cuda.empty_cache()


## 5. Language Embeddings (Qwen3-1.7B)

Extract per-layer embeddings from AudioCaps captions using Qwen3-1.7B (29 layers, 2048-dim).
Embeddings are attention-masked mean-pooled over tokens.


In [ ]:
qwen3_tokenizer, qwen3_model = load_qwen3_model("Qwen/Qwen3-1.7B")

extract_qwen3_embeddings(
    df=df,
    text_col="caption",
    save_root="qwen3-1.7b/text_embeddings",
    tokenizer=qwen3_tokenizer,
    model=qwen3_model,
    device=device,
)
get_layer_embeddings(
    input_folder="qwen3-1.7b/text_embeddings",
    save_root="qwen3-1.7b/text_layers",
    num_layers=29, emb_shape=(29, 2048),
)

del qwen3_model; gc.collect(); torch.cuda.empty_cache()


## 6. Verify Embeddings

Confirm all models produced embeddings with the expected shapes and that sample IDs are aligned across modalities.


In [ ]:
checks = {
    "DINOv2 Large – frames":    ("dinov2/large/frame_layers",        25, (25, 1024)),
    "DINOv2 Large – gen":       ("dinov2/large/gen_layers",          25, (25, 1024)),
    "DINOv2 Base – frames":     ("dinov2/base/frame_layers",         13, (13, 768)),
    "DINOv2 Base – gen":        ("dinov2/base/gen_layers",           13, (13, 768)),
    "BEATs Iter3":              ("beats/iter3/audio_layers",         12, (12, 768)),
    "BEATs Iter3+":             ("beats/iter3_plus/audio_layers",    12, (12, 768)),
    "Qwen3-1.7B":               ("qwen3-1.7b/text_layers",           29, (29, 2048)),
}

for name, (folder, num_layers, _) in checks.items():
    ids_path = os.path.join(folder, "ids.npy")
    layer0   = os.path.join(folder, "layer_0.npy")
    if not os.path.exists(ids_path):
        print(f"  MISSING: {name} — ids.npy not found in {folder}")
        continue
    ids   = np.load(ids_path)
    layer = np.load(layer0)
    files = [f for f in os.listdir(folder) if f.startswith("layer_")]
    print(f"  {name}: {len(ids)} samples | layer_0 shape: {layer.shape} | {len(files)} layer files")


In [ ]:
# Confirm IDs are aligned across all modalities
import itertools

id_sets = {
    name: set(np.load(os.path.join(folder, "ids.npy")))
    for name, (folder, _, _) in checks.items()
    if os.path.exists(os.path.join(folder, "ids.npy"))
}

common_ids = set.intersection(*id_sets.values())
print(f"Common IDs across all modalities: {len(common_ids)}")

for (a, s1), (b, s2) in itertools.combinations(id_sets.items(), 2):
    diff = len(s1.symmetric_difference(s2))
    if diff:
        print(f"  ⚠ {a} vs {b}: {diff} IDs differ")


## 7. Archive Layer Embeddings

Zip the layer embedding folders for download or transfer to another environment.
These `.zip` files are the inputs for the alignment analysis notebook.


In [ ]:
archives = {
    "dinov2_large_frame_layers": "dinov2/large/frame_layers",
    "dinov2_large_gen_layers":   "dinov2/large/gen_layers",
    "dinov2_base_frame_layers":  "dinov2/base/frame_layers",
    "dinov2_base_gen_layers":    "dinov2/base/gen_layers",
    "beats_iter3_layers":        "beats/iter3/audio_layers",
    "beats_iter3_plus_layers":   "beats/iter3_plus/audio_layers",
    "qwen3_1.7b_layers":         "qwen3-1.7b/text_layers",
}

for archive_name, folder in archives.items():
    if os.path.exists(folder):
        shutil.make_archive(archive_name, "zip", folder)
        size_mb = os.path.getsize(f"{archive_name}.zip") / 1e6
        print(f"  {archive_name}.zip  ({size_mb:.1f} MB)")
    else:
        print(f"  Skipped {archive_name} — folder not found")
